In [2]:
from pyspark.sql.functions import col, date_format

# 1. Ler as duas tabelas
df_gold = spark.read.table("gold_hourly_metrics")
df_weather = spark.read.table("silver_weather")

# 2. Preparar a silver_weather
df_weather = df_weather.withColumn("event_timestamp", date_format(col("event_timestamp"), "yyyy-MM-dd'T'HH:mm:ss"))
df_weather = df_weather.withColumnRenamed("event_timestamp", "datetime")

# 3. JOIN
df_gold_weather = df_gold.join(df_weather, "datetime", "left")

# 4. Guardar
df_gold_weather.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable("gold_hourly_weather")

print(f"Gold weather: {df_gold_weather.count()} registos")


StatementMeta(, 47fd1820-6f99-41a9-8092-d2fc9d3898a4, 4, Finished, Available, Finished, False)

Gold weather: 14564 registos


In [3]:
display(df_gold_weather)

StatementMeta(, 47fd1820-6f99-41a9-8092-d2fc9d3898a4, 5, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, edd53434-d6b3-4a4d-ae4c-7601a9701f65)

In [1]:
# 1. Ler a silver
df_forecast = spark.read.table("silver_weather_forecast")

# 2. Guardar como Gold
df_forecast.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable("gold_weather_forecast")

print(f"Gold weather forecast: {df_forecast.count()} registos")

StatementMeta(, f50b149f-a2ce-4538-9c68-47312ada3572, 3, Finished, Available, Finished, False)

Gold weather forecast: 192 registos


In [1]:
spark.sql("""
    SELECT event_timestamp, temperature_ext, humidity_ext
    FROM silver_weather_forecast
    ORDER BY event_timestamp DESC
    LIMIT 10
""").show()

StatementMeta(, 15d84672-d137-4463-9ea1-aec80a81ca53, 3, Finished, Available, Finished, False)

+-------------------+---------------+------------+
|    event_timestamp|temperature_ext|humidity_ext|
+-------------------+---------------+------------+
|2026-04-27 23:00:00|           11.4|        91.0|
|2026-04-27 22:00:00|           11.7|        89.0|
|2026-04-27 21:00:00|           12.8|        84.0|
|2026-04-27 20:00:00|           15.3|        73.0|
|2026-04-27 19:00:00|           18.7|        59.0|
|2026-04-27 18:00:00|           21.5|        46.0|
|2026-04-27 17:00:00|           23.6|        36.0|
|2026-04-27 16:00:00|           25.1|        28.0|
|2026-04-27 15:00:00|           26.0|        23.0|
|2026-04-27 14:00:00|           26.1|        23.0|
+-------------------+---------------+------------+

